# EH ADBE Agent Host 
## Important Instructions
- Do not run this notebook directly
- Keep it fresh (clear outputs and execution history)

## About
This notebook is a template used by the backend to start an agentic workflow on cloud GPUs.

This specific notebook is for image generation workflows.

In [ ]:
# Injection template
BACKEND_WS_URL = ""

# Injection space


In [ ]:
!pip install -q diffusers transformers accelerate safetensors websockets

In [ ]:
import asyncio
import copy
import gc
import io
import random
from concurrent.futures import ThreadPoolExecutor

import torch
import websockets
from diffusers import StableDiffusionXLPipeline
from PIL import Image

MODEL_ID = "stabilityai/sdxl-turbo"
BATCH_SIZE = 4

def load_pipelines():
    """Load weights once, then place independent copies on cuda:0 and cuda:1."""
    print("Loading SDXL...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.vae.enable_slicing()
    pipe1 = copy.deepcopy(pipe)
    pipe0 = pipe.to("cuda:0")
    pipe1 = pipe1.to("cuda:1")
    return pipe0, pipe1


pipes = None
def release_gpu_memory():
    """Unload prior pipelines (e.g. re-run cell) and return VRAM to the driver."""
    global pipes
    del pipes
    gc.collect()
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
    torch.cuda.ipc_collect()



async def generate(pipes, prompt):
    q = asyncio.Queue()
    seeds = [random.randint(0, 2**32 - 1) for _ in range(2)]
    loop = asyncio.get_running_loop()

    def generate_image(pipe, device, seed):
        generator = torch.Generator(device=device).manual_seed(seed)
        images = pipe(
            prompt=prompt, 
            generator=generator,
            num_images_per_prompt=BATCH_SIZE
        ).images

        for image in images:
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            loop.call_soon_threadsafe(q.put_nowait, buf.getvalue())
        loop.call_soon_threadsafe(q.put_nowait, None)

    asyncio.create_task(asyncio.to_thread(generate_image, pipes[0], "cuda:0", seeds[0]))
    asyncio.create_task(asyncio.to_thread(generate_image, pipes[1], "cuda:1", seeds[1]))
    
    done = 0
    sent = 0
    while True:
        image = await q.get()
        if image is None:
            done += 1
            print("Done", done)
            if done == 2:
                break
            continue
        sent += 1
        print("Sent", sent)
        yield image
    

async def run():
    global pipes
    release_gpu_memory()

    try:
        pipes = load_pipelines()

        async with websockets.connect(
            BACKEND_WS_URL,
            max_size=None,
            ping_interval=30,
            ping_timeout=300,
        ) as ws:
            print("Connected to backend")

            while True:
                msg = await ws.recv()
                if msg == "DONE":
                    break

                print("Generating...")
                async for image in generate(pipes, msg):
                    await ws.send(image)
                await ws.send(b"")
                print("Sent images")
    finally:
        release_gpu_memory()


await run()